## 第四章 自己动手：压缩一个流形，然后找到它的账单

> 沿着作者设计的五个步骤，用 Swiss Roll 数据集亲手验证流形压缩的代价。

---

### 第一步：数据生成

**问题 1（写代码前先回答）：这个数据的"真实内在维度"是多少？**

Swiss Roll 由两个独立参数生成：
- `t`：卷轴角度（同时等于水平面极径，决定在卷轴上的位置）
- `h`：高度（与 t 完全独立）

所以**真实内在维度 = 2**，嵌入在三维空间里。

**PCA 预测**：PCA 是基于线性假设的方法，而 Swiss Roll 的 x、y 坐标由 t 的余弦/正弦生成（非线性）。即使内在维度是 2，PCA 也可能需要 3 个主成分才能接近 100% 方差——因为它无法"折叠"非线性卷轴结构。

**等一会儿和 PCA 实际结果对比。**

In [ ]:
import numpy as np
# import matplotlib.pyplot as plt

# plt.rcParams['font.family'] = ['PingFang HK']  # 或 'Arial Unicode MS'\'PingFang HK'
# plt.rcParams['axes.unicode_minus'] = False  # 修复负号显示问题

In [ ]:
from sklearn.datasets import make_swiss_roll

np.random.seed(42)
N = 2000  # 采样点数

# 使用 sklearn 生成标准 Swiss Roll
# t 是内在坐标（卷轴角度），z 是高度方向的内在坐标
data_swiss, t_swiss = make_swiss_roll(n_samples=N, noise=0.1, random_state=42)
# data_swiss shape=(N, 3)，列分别是 x, h, y（三维嵌入坐标）


In [ ]:
x_sw = data_swiss[:, 0]   # t · cos(t)
h_sw = data_swiss[:, 1]   # 高度 h（内在坐标 2），与 t 无关
y_sw = data_swiss[:, 2]   # t · sin(t)

# 验证 t ≈ sqrt(x² + y²)：随机抽 100 个点对比
rng = np.random.default_rng(0)
idx_sample = rng.choice(len(t_swiss), 100, replace=False)

t_recovered = np.sqrt(x_sw**2 + y_sw**2)
errors = np.abs(t_recovered[idx_sample] - t_swiss[idx_sample])

# 打印前 10 行作为样本
print(f"{'点索引':>6}  {'t_swiss':>10}  {'√(x²+y²)':>10}  {'|误差|':>8}")
print("─" * 42)
for i in idx_sample[:10]:
    err = abs(t_recovered[i] - t_swiss[i])
    print(f"{i:>6}  {t_swiss[i]:>10.4f}  {t_recovered[i]:>10.4f}  {err:>8.4f}")
print(f"  ...")

# 100 点统计
print(f"\n100 个随机样本统计（noise=0.1 引入的扰动）：")
print(f"  平均绝对误差：{errors.mean():.4f}")
print(f"  最大绝对误差：{errors.max():.4f}")
print(f"  误差标准差：  {errors.std():.4f}")


#### Swiss Roll 的坐标结构

`make_swiss_roll` 返回两个东西：
- **`data_swiss`**（shape `N×3`）：三维嵌入坐标
- **`t_swiss`**（shape `N`）：内在坐标 `t`（卷轴角度）

三列的含义：

| 列索引 | 变量 | 公式 | 含义 |
|:---:|:---:|:---:|:---|
| `[:, 0]` | $x$ | $t\cos t$ | 水平面投影的 x 分量 |
| `[:, 1]` | $h$ | 均匀随机 | **高度**，第二个内在坐标，与 t 无关 |
| `[:, 2]` | $y$ | $t\sin t$ | 水平面投影的 y 分量 |

**关键关系**：$t = \sqrt{x^2 + y^2}$（水平面极径 = 卷轴角度）

这也是为什么测试要用 `col[0]` 和 `col[2]`，而不是 `col[0]` 和 `col[1]`：col[1] 是高度，  
与 t 完全独立，放进去会破坏这个几何关系。

加了 `noise=0.1` 之后坐标有小扰动，所以 $\sqrt{x^2+y^2} \approx t$ 而非精确相等。

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "scatter3d"}, {"type": "scatter3d"}]],
    subplot_titles=[
        "按内在坐标 t 染色（第几圈）",
        "按高度 h 染色（第二内在坐标）",
    ],
    horizontal_spacing=0.02,
)

fig.add_trace(go.Scatter3d(
    x=x_sw, y=h_sw, z=y_sw, mode='markers',
    marker=dict(
        size=2, opacity=0.75,
        color=t_swiss, colorscale='Viridis',
        colorbar=dict(title=dict(text="t"), len=0.65, x=0.42, thickness=14),
    ),
    name='t',
), row=1, col=1)

fig.add_trace(go.Scatter3d(
    x=x_sw, y=h_sw, z=y_sw, mode='markers',
    marker=dict(
        size=2, opacity=0.75,
        color=h_sw, colorscale='Plasma',
        colorbar=dict(title=dict(text="h"), len=0.65, x=1.02, thickness=14),
    ),
    name='高度 h',
), row=1, col=2)

_axis = dict(
    xaxis=dict(title="x = t·cos(t)"),
    yaxis=dict(title="h（高度）"),
    zaxis=dict(title="y = t·sin(t)"),
)

fig.update_layout(
    # 左图：从 h 轴方向看过去，x-h 平面与视线接近平行，z 轴朝上
    scene=dict(
        **_axis,
        camera=dict(
            eye=dict(x=0.08, y=2.2, z=0.06),
            up=dict(x=0, y=0, z=1),
            center=dict(x=0, y=0, z=0),
        ),
    ),
    # 右图：从 x-z 平面斜看，up 沿 h(y) 轴方向使高度轴朝上
    scene2=dict(
        **_axis,
        camera=dict(
            eye=dict(x=1.6, y=0.25, z=1.2),
            up=dict(x=0, y=1, z=0),
            center=dict(x=0, y=0, z=0),
        ),
    ),
    height=600,
    showlegend=False,
    title=dict(
        text=(
            "Swiss Roll：三维嵌入坐标 vs 内在坐标<br>"
            "<sup>左图 t 沿螺旋径向连续渐变；右图 h 竖向分层，与 t 无关</sup>"
        ),
        x=0.5,
        font=dict(size=16),
    ),
)
fig.show()


In [ ]:
# 后续使用 sklearn 版本（更标准）
data = data_swiss          # 三维坐标，shape=(N, 3)
intrinsic_t = t_swiss      # 内在坐标 t，用于染色

In [ ]:
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, TensorDataset

# 训练集 80% / 测试集 20%
torch.manual_seed(42)
N     = len(data)
split = int(0.8 * N)

train_np   = data[:split]   # 原始坐标，shape=(1600, 3)
test_np    = data[split:]   # 原始坐标，shape=(400, 3)
scaler = StandardScaler().fit(train_np)
train_scaled_np = scaler.transform(train_np)
test_scaled_np  = scaler.transform(test_np)
train_data = torch.tensor(train_scaled_np, dtype=torch.float32)
test_data  = torch.tensor(test_scaled_np,  dtype=torch.float32)
train_t    = intrinsic_t[:split]   # 内在坐标 t，用于潜在空间染色

print(f"训练集：{train_np.shape}，测试集：{test_np.shape}")
print(f"内在坐标 t 范围：[{intrinsic_t.min():.2f}, {intrinsic_t.max():.2f}]")
print("已对 3D 输入按训练集统计量做标准化；后续可视化会再映射回原始坐标。")

---

### 第二步：实现统一的主实验自动编码器

结构：**3 → 64 → 64 → d（瓶颈）→ 64 → 64 → 3**

所有 `d=1,2,3,4,5` 共用**同一套更强的模型与训练流程**，只改变瓶颈维度。这样比较才公平。  
输入先按训练集统计量标准化，再送进自动编码器；评估和画图时再把重建结果反标准化回原始三维坐标。  
瓶颈维度 d 仍然是"压缩程度"的旋钮——d 越小，信息越被迫丢弃。

> **流形视角**：编码器在学习流形的坐标参数化，解码器在学习从坐标恢复嵌入坐标。

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, bottleneck_dim, hidden_dim=64):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(3, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, bottleneck_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 3)
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z


def train_autoencoder(train_tensor, bottleneck_dim, n_epochs=2500, lr=1e-3, batch_size=256):
    loader    = DataLoader(TensorDataset(train_tensor), batch_size=batch_size, shuffle=True)
    model     = Autoencoder(bottleneck_dim)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=150, min_lr=1e-5)
    loss_fn   = nn.MSELoss()
    best_loss = float('inf')
    best_state = copy.deepcopy(model.state_dict())

    for epoch in range(n_epochs):
        epoch_loss = 0.0
        for (batch,) in loader:
            optimizer.zero_grad()
            x_hat, _ = model(batch)
            loss = loss_fn(x_hat, batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(batch)

        epoch_loss /= len(train_tensor)
        scheduler.step(epoch_loss)

        if epoch_loss < best_loss:
            best_loss = epoch_loss
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    return model


print("统一主实验自动编码器定义完成")

---

### 第三步：系统扫描瓶颈维度 d = 1, 2, 3, 4, 5

对每个 d，训练一个自动编码器，记录：
- 训练集 / 测试集重建误差（MSE）
- `d=1,2`：左图看训练集潜在空间（t 染色），右图直接看**测试集原始点 vs 测试集重建点**
- `d=3`：左图看 3D 潜在空间，右图同样比较**测试集原始点 vs 测试集重建点**
- `d=4,5`：先只看 MSE 数字

这里先把 **reconstruction（分布内泛化）** 和 **decoder-only 随机采样** 分开。随机采样 latent 的实验会在下一小节单独做，避免和测试集重建混在一起。


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

results = {}

for d in [1, 2, 3, 4, 5]:
    print(f"训练 d={d} ...", end=' ', flush=True)
    model = train_autoencoder(train_data, bottleneck_dim=d, n_epochs=2500)
    model.eval()

    with torch.no_grad():
        train_hat, train_z = model(train_data)
        test_hat,  test_z  = model(test_data)
        train_mse = nn.MSELoss()(train_hat, train_data).item()
        test_mse  = nn.MSELoss()(test_hat,  test_data).item()

    train_hat_np = scaler.inverse_transform(train_hat.numpy())
    test_hat_np  = scaler.inverse_transform(test_hat.numpy())

    results[d] = {
        'train_mse': train_mse,
        'test_mse': test_mse,
        'train_z': train_z.numpy(),
        'test_z': test_z.numpy(),
        'train_hat': train_hat_np,
        'test_hat': test_hat_np,
        'model': model,
    }
    print(f"训练MSE={train_mse:.4f}，测试MSE={test_mse:.4f}")

    if d > 3:
        continue   # d=4,5 先只看 MSE 数字

    z_np = results[d]['train_z']
    test_hat_np = results[d]['test_hat']

    # ── d=1 或 d=2：左=训练集潜在空间，右=测试集原始 vs 重建 ──────────
    if d <= 2:
        fig = make_subplots(
            rows=1, cols=2,
            specs=[[{"type": "scatter"}, {"type": "scatter3d"}]],
            subplot_titles=[
                f"d={d} 训练集潜在空间（t 染色）<br><sup>看编码器是否学到平滑坐标，而不是直接评价泛化</sup>",
                f"d={d} 测试集：原始点 vs 重建点<br><sup>蓝色 = test_data，橙色 = test_hat，直接看 held-out reconstruction</sup>",
            ],
            horizontal_spacing=0.06,
        )

        cb = dict(title=dict(text="t"), x=0.42, len=0.75, thickness=12)
        if d == 1:
            fig.add_trace(go.Scatter(
                x=z_np[:, 0], y=np.zeros(len(z_np)),
                mode='markers',
                marker=dict(size=3, color=train_t, colorscale='Viridis', colorbar=cb),
                name='训练集潜在点（t 染色）',
            ), row=1, col=1)
            fig.update_yaxes(visible=False, row=1, col=1)
            fig.update_xaxes(title_text="z₁", row=1, col=1)
        else:
            fig.add_trace(go.Scatter(
                x=z_np[:, 0], y=z_np[:, 1],
                mode='markers',
                marker=dict(size=3, color=train_t, colorscale='Viridis', colorbar=cb),
                name='训练集潜在点（t 染色）',
            ), row=1, col=1)
            fig.update_xaxes(title_text="z₁", row=1, col=1)
            fig.update_yaxes(title_text="z₂", row=1, col=1)

        fig.add_trace(go.Scatter3d(
            x=test_np[:, 0], y=test_np[:, 1], z=test_np[:, 2],
            mode='markers',
            marker=dict(size=1.8, color='lightsteelblue', opacity=0.28),
            name='测试集原始点',
        ), row=1, col=2)
        fig.add_trace(go.Scatter3d(
            x=test_hat_np[:, 0], y=test_hat_np[:, 1], z=test_hat_np[:, 2],
            mode='markers',
            marker=dict(size=2.2, color='darkorange', opacity=0.78),
            name='测试集重建点',
        ), row=1, col=2)

        fig.update_layout(
            scene=dict(
                xaxis=dict(title="x = t·cos(t)"),
                yaxis=dict(title="h（高度）"),
                zaxis=dict(title="y = t·sin(t)"),
            ),
            height=520,
            showlegend=True,
            legend=dict(x=0.5, y=0.52, xanchor='center', yanchor='middle'),
            title=dict(text=f"瓶颈维度 d={d}", x=0.5, font=dict(size=14)),
        )

    # ── d=3：左=3D潜在空间，右=测试集原始 vs 重建 ─────────────────
    else:
        fig = make_subplots(
            rows=1, cols=2,
            specs=[[{"type": "scatter3d"}, {"type": "scatter3d"}]],
            subplot_titles=[
                "d=3 潜在空间（z₁×z₂×z₃，t 染色）<br><sup>d=3 时更容易学近似恒等变换，潜在几何仅供参考</sup>",
                "d=3 测试集：原始点 vs 重建点<br><sup>看 held-out reconstruction 是否已接近贴合</sup>",
            ],
            horizontal_spacing=0.04,
        )

        fig.add_trace(go.Scatter3d(
            x=z_np[:, 0], y=z_np[:, 1], z=z_np[:, 2],
            mode='markers',
            marker=dict(size=2, color=train_t, colorscale='Viridis', colorbar=dict(title=dict(text="t"), x=0.42, len=0.75, thickness=12)),
            name='训练集潜在点',
        ), row=1, col=1)

        fig.add_trace(go.Scatter3d(
            x=test_np[:, 0], y=test_np[:, 1], z=test_np[:, 2],
            mode='markers',
            marker=dict(size=1.8, color='lightsteelblue', opacity=0.25),
            name='测试集原始点',
        ), row=1, col=2)
        fig.add_trace(go.Scatter3d(
            x=test_hat_np[:, 0], y=test_hat_np[:, 1], z=test_hat_np[:, 2],
            mode='markers',
            marker=dict(size=2.2, color='darkorange', opacity=0.8),
            name='测试集重建点',
        ), row=1, col=2)

        _ax = dict(xaxis=dict(title="z₁"), yaxis=dict(title="z₂"), zaxis=dict(title="z₃"))
        _ax_orig = dict(xaxis=dict(title="x"), yaxis=dict(title="h"), zaxis=dict(title="y"))
        fig.update_layout(
            scene=_ax,
            scene2=_ax_orig,
            height=520,
            showlegend=True,
            legend=dict(x=0.5, y=0.52, xanchor='center', yanchor='middle'),
            title=dict(text="瓶颈维度 d=3", x=0.5, font=dict(size=14)),
        )

    fig.show()

# ── 误差汇总表 ────────────────────────────────────────────────────
print(f"\n{'d':>4} | {'训练MSE':>10} | {'测试MSE':>10}")
print('-' * 32)
for d, r in results.items():
    print(f"{d:>4} | {r['train_mse']:>10.4f} | {r['test_mse']:>10.4f}")


---

**问题 2 的回答：d=1 时保留了什么，丢失了什么？**

Swiss Roll 有两个内在坐标：`t`（卷轴角度）和 `h`（高度），d=1 只剩一个维度。

- **保留**：`t` 方向的排序——z₁ 沿 t 方向连续变化（诊断图：z₁ vs h 散点中颜色横向渐变）
- **丢失**：`h` 高度信息——高度不同但 t 相近的点被映射到几乎相同的 z₁，重建时产生"折叠"（诊断图：同一 z₁ 位置对应所有高度值，颜色竖向散布）
- **拓扑**：Swiss Roll 的内在拓扑是矩形（t × h 两个区间，没有闭合环），所以 d=1 的线段表示在拓扑上兼容——问题是维度不足，不是拓扑破坏

---

**问题 3 的回答：d=2 时潜在空间颜色是否连续？**

d=2 的潜在空间用 `t` 染色呈**连续渐变**，说明：
- 编码器学到了保持 t 方向局部结构的坐标变换
- 两个潜在维度分别对应了 t 和 h 的某种变换（诊断图 h 染色中可见分层）
- 这就是"统计约束把向量推到流形上"的直观呈现

> **补充**：z₁, z₂ 不等于 (t, h)，而是它们的某个非线性变换——编码器只关心能否重建，不关心潜在坐标是否"可解释"。

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 需要 results[1] 和 results[2] 已训练完毕
z1_np = results[1]['train_z']   # shape=(1600, 1)
z2_np = results[2]['train_z']   # shape=(1600, 2)
h_train = train_np[:, 1]        # 高度 h

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        "d=1：z₁ vs h（t 染色）<br>"
        "<sup>证实：颜色沿 z₁ 横向渐变 = z₁ 编码 t；同一 z₁ 竖向散布各种 h = h 被折叠丢弃</sup>",
        "d=2：z₁ vs z₂（t 染色）<br>"
        "<sup>两个维度学到的结构，观察是否一个轴对应 t、另一个对应 h</sup>",
    ],
    horizontal_spacing=0.12,
)

cb = dict(title=dict(text="t"), thickness=12, len=0.8)

# 左图：d=1，z₁ vs h
fig.add_trace(go.Scatter(
    x=z1_np[:, 0], y=h_train,
    mode='markers',
    marker=dict(size=3, color=train_t, colorscale='Viridis',
                colorbar=dict(**cb, x=0.38)),
    name='d=1',
), row=1, col=1)
fig.update_xaxes(title_text="z₁（唯一潜在维度）", row=1, col=1)
fig.update_yaxes(title_text="h（真实高度）", row=1, col=1)

# 右图：d=2，z₁ vs z₂（同时叠加 h 用点大小编码）
fig.add_trace(go.Scatter(
    x=z2_np[:, 0], y=z2_np[:, 1],
    mode='markers',
    marker=dict(size=3, color=train_t, colorscale='Viridis',
                colorbar=dict(**cb, x=1.01)),
    name='d=2',
), row=1, col=2)
fig.update_xaxes(title_text="z₁", row=1, col=2)
fig.update_yaxes(title_text="z₂", row=1, col=2)

fig.update_layout(
    height=440, showlegend=False,
    title=dict(
        text="d=1 折叠诊断（已证实）：z₁ 编码 t，h 信息丢失",
        x=0.5, font=dict(size=13),
    ),
)
fig.show()

# 补充：用 h 染色再看一遍，验证 d=1 的 z₁ 与 h 无关
fig2 = make_subplots(rows=1, cols=2,
    subplot_titles=[
        "d=1：z₁ vs h（h 染色）<br><sup>如果 h 信息丢失，颜色应该与 z₁ 无关联</sup>",
        "d=2：z₁ vs z₂（h 染色）<br><sup>观察哪个维度对应 h</sup>",
    ],
    horizontal_spacing=0.12,
)

cb_h = dict(title=dict(text="h"), thickness=12, len=0.8)
fig2.add_trace(go.Scatter(
    x=z1_np[:, 0], y=h_train, mode='markers',
    marker=dict(size=3, color=h_train, colorscale='Plasma',
                colorbar=dict(**cb_h, x=0.38)),
), row=1, col=1)
fig2.update_xaxes(title_text="z₁", row=1, col=1)
fig2.update_yaxes(title_text="h", row=1, col=1)

fig2.add_trace(go.Scatter(
    x=z2_np[:, 0], y=z2_np[:, 1], mode='markers',
    marker=dict(size=3, color=h_train, colorscale='Plasma',
                colorbar=dict(**cb_h, x=1.01)),
), row=1, col=2)
fig2.update_xaxes(title_text="z₁", row=1, col=2)
fig2.update_yaxes(title_text="z₂", row=1, col=2)

fig2.update_layout(
    height=440, showlegend=False,
    title=dict(
        text="用 h 染色（已证实）：d=2 中某个潜在维度对应 h",
        x=0.5, font=dict(size=13),
    ),
)
fig2.show()


---

### 第三步补充：把 reconstruction 和 random decode 分开看

上面右图回答的是：**测试集点经过 encoder→decoder 后，能否被重建回来？**

这里单独做另一个问题：**如果不经过 encoder，只在潜在空间里随机采样再交给 decoder，会发生什么？**

这一步不是泛化测试，而是检查普通自动编码器的 latent 是否适合直接采样。


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

for d in [1, 2, 3]:
    model = results[d]['model']
    z_np = results[d]['train_z']

    with torch.no_grad():
        z_min = torch.tensor(z_np.min(axis=0), dtype=torch.float32)
        z_max = torch.tensor(z_np.max(axis=0), dtype=torch.float32)
        z_rand = torch.rand(200, d) * (z_max - z_min) + z_min
        decoded = scaler.inverse_transform(model.decoder(z_rand).numpy())

    if d <= 2:
        fig = make_subplots(
            rows=1, cols=2,
            specs=[[{"type": "scatter"}, {"type": "scatter3d"}]],
            subplot_titles=[
                f"d={d} latent：实际占据区域 vs 随机采样<br><sup>蓝点 = encoder 学到的支持集；红点 = 包围盒随机采样的 z_rand</sup>",
                f"d={d} random latent decode<br><sup>红点 = 从包围盒随机采样 z 再解码，可能落到流形外</sup>",
            ],
            horizontal_spacing=0.06,
        )

        if d == 1:
            fig.add_trace(go.Scatter(
                x=z_np[:, 0], y=np.zeros(len(z_np)),
                mode='markers',
                marker=dict(size=3, color='steelblue', opacity=0.75),
                name='训练集 latent',
            ), row=1, col=1)
            fig.add_trace(go.Scatter(
                x=z_rand[:, 0].numpy(), y=np.zeros(len(z_rand)),
                mode='markers',
                marker=dict(size=3, color='tomato', opacity=0.55),
                name='随机采样 latent',
            ), row=1, col=1)
            fig.update_yaxes(visible=False, row=1, col=1)
            fig.update_xaxes(title_text="z₁", row=1, col=1)
        else:
            fig.add_trace(go.Scatter(
                x=z_np[:, 0], y=z_np[:, 1],
                mode='markers',
                marker=dict(size=3, color='steelblue', opacity=0.75),
                name='训练集 latent',
            ), row=1, col=1)
            fig.add_trace(go.Scatter(
                x=z_rand[:, 0].numpy(), y=z_rand[:, 1].numpy(),
                mode='markers',
                marker=dict(size=3, color='tomato', opacity=0.55),
                name='随机采样 latent',
            ), row=1, col=1)
            fig.update_xaxes(title_text="z₁", row=1, col=1)
            fig.update_yaxes(title_text="z₂", row=1, col=1)

        fig.add_trace(go.Scatter3d(
            x=train_np[:, 0], y=train_np[:, 1], z=train_np[:, 2],
            mode='markers',
            marker=dict(size=1.5, color='lightsteelblue', opacity=0.22),
            name='训练流形参考',
        ), row=1, col=2)
        fig.add_trace(go.Scatter3d(
            x=decoded[:, 0], y=decoded[:, 1], z=decoded[:, 2],
            mode='markers',
            marker=dict(size=2.1, color='tomato', opacity=0.78),
            name='随机解码点',
        ), row=1, col=2)

        fig.update_layout(
            scene=dict(
                xaxis=dict(title="x = t·cos(t)"),
                yaxis=dict(title="h（高度）"),
                zaxis=dict(title="y = t·sin(t)"),
            ),
            height=520,
            showlegend=True,
            legend=dict(x=0.5, y=0.52, xanchor='center', yanchor='middle'),
            title=dict(text=f"d={d}：decoder-only 随机采样实验", x=0.5, font=dict(size=14)),
        )
    else:
        fig = make_subplots(
            rows=1, cols=2,
            specs=[[{"type": "scatter3d"}, {"type": "scatter3d"}]],
            subplot_titles=[
                "d=3 latent：实际占据区域 vs 随机采样",
                "d=3 random latent decode",
            ],
            horizontal_spacing=0.04,
        )

        fig.add_trace(go.Scatter3d(
            x=z_np[:, 0], y=z_np[:, 1], z=z_np[:, 2],
            mode='markers',
            marker=dict(size=2, color='steelblue', opacity=0.75),
            name='训练集 latent',
        ), row=1, col=1)
        fig.add_trace(go.Scatter3d(
            x=z_rand[:, 0].numpy(), y=z_rand[:, 1].numpy(), z=z_rand[:, 2].numpy(),
            mode='markers',
            marker=dict(size=2, color='tomato', opacity=0.55),
            name='随机采样 latent',
        ), row=1, col=1)
        fig.add_trace(go.Scatter3d(
            x=train_np[:, 0], y=train_np[:, 1], z=train_np[:, 2],
            mode='markers',
            marker=dict(size=1.5, color='lightsteelblue', opacity=0.22),
            name='训练流形参考',
        ), row=1, col=2)
        fig.add_trace(go.Scatter3d(
            x=decoded[:, 0], y=decoded[:, 1], z=decoded[:, 2],
            mode='markers',
            marker=dict(size=2.1, color='tomato', opacity=0.78),
            name='随机解码点',
        ), row=1, col=2)

        fig.update_layout(
            scene=dict(xaxis=dict(title="z₁"), yaxis=dict(title="z₂"), zaxis=dict(title="z₃")),
            scene2=dict(xaxis=dict(title="x"), yaxis=dict(title="h"), zaxis=dict(title="y")),
            height=520,
            showlegend=True,
            legend=dict(x=0.5, y=0.52, xanchor='center', yanchor='middle'),
            title=dict(text="d=3：decoder-only 随机采样实验", x=0.5, font=dict(size=14)),
        )

    fig.show()


---

#### 为什么随机采样解码点会偏离参考流形

左图里蓝点表示 **encoder 真正学到、并被训练数据实际占据的 latent 支持集**；红点表示我们在逐维包围盒里均匀随机采样得到的 `z_rand`。这两者通常并不重合。

普通自动编码器只被要求：**把训练样本编码到某些 z，再把这些 z 解码回来**。它从来没有被要求让“包围盒里的每个点”都对应真实流形。于是：
- latent 支持集可能是弯曲的、细长的，甚至带空洞
- 包围盒均匀采样会落到许多训练从未覆盖过的区域
- decoder 对这些 unsupported 的 z 只能外推（extrapolation），所以右图很多红点会偏离 Swiss Roll

这和 [chapter4](/Volumes/Samsung990Pro/KnowledgeHub/AI_reasoning/reasoning-kingdom/docs/volume1/chapter4/index.md:263) 到 [271](/Volumes/Samsung990Pro/KnowledgeHub/AI_reasoning/reasoning-kingdom/docs/volume1/chapter4/index.md:271) 讨论的“**流形外的点**”是同一类现象，只是这里的 OOD 发生在 **潜在空间**，不是输入空间。书里的表述是：模型没有“这个点不在我的流形上”的警报，只能把点拉向它学过的结构并给出一个看似合理的答案。随机 latent 解码图就是这个机制的一个可视化版本。

如果我们希望 **随机采样 latent 也更有依据**，就需要给潜在空间额外的分布约束。VAE 就是在做这件事。

---

#### `d=2` 的 VAE 采样对比

下面单独训练一个 `d=2` 的 VAE。它在重建损失之外加入 KL 散度，把后验 `q(z|x)` 拉向标准正态 `N(0, I)`。

这不是为了让 VAE 在重建上一定优于普通 AE，而是为了比较：
- 普通 AE：latent 没有规则先验，包围盒随机采样容易落进空洞
- VAE：latent 被规范到接近标准正态，`torch.randn(...)` 采样更有意义


In [ ]:
import copy
import random
import plotly.graph_objects as go
from plotly.subplots import make_subplots

class VariationalAutoencoder(nn.Module):
    def __init__(self, latent_dim=2, hidden_dim=64):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(3, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
        )
        self.mu_head = nn.Linear(hidden_dim, latent_dim)
        self.logvar_head = nn.Linear(hidden_dim, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 3),
        )

    def encode(self, x):
        h = self.backbone(x)
        mu = self.mu_head(h)
        logvar = self.logvar_head(h)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decoder(z)
        return x_hat, mu, logvar, z


def train_vae(train_tensor, latent_dim=2, n_epochs=2500, lr=1e-3, batch_size=256, beta=1e-3, seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    g = torch.Generator()
    g.manual_seed(seed)

    loader = DataLoader(TensorDataset(train_tensor), batch_size=batch_size, shuffle=True, generator=g)
    model = VariationalAutoencoder(latent_dim=latent_dim)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=150, min_lr=1e-5)
    recon_loss_fn = nn.MSELoss(reduction='mean')

    best_loss = float('inf')
    best_state = copy.deepcopy(model.state_dict())

    for epoch in range(n_epochs):
        epoch_loss = 0.0
        for (batch,) in loader:
            optimizer.zero_grad()
            x_hat, mu, logvar, _ = model(batch)
            recon_loss = recon_loss_fn(x_hat, batch)
            kl_loss = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
            loss = recon_loss + beta * kl_loss
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(batch)

        epoch_loss /= len(train_tensor)
        scheduler.step(epoch_loss)

        if epoch_loss < best_loss:
            best_loss = epoch_loss
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    return model


vae_d2 = train_vae(train_data, latent_dim=2, n_epochs=2500, beta=1e-3)
vae_d2.eval()

with torch.no_grad():
    train_hat_vae, train_mu_vae, train_logvar_vae, _ = vae_d2(train_data)
    test_hat_vae, test_mu_vae, test_logvar_vae, _ = vae_d2(test_data)
    vae_train_mse = nn.MSELoss()(train_hat_vae, train_data).item()
    vae_test_mse = nn.MSELoss()(test_hat_vae, test_data).item()
    z_prior = torch.randn(200, 2)
    sampled_vae = scaler.inverse_transform(vae_d2.decoder(z_prior).numpy())

train_mu_np = train_mu_vae.numpy()
test_hat_vae_np = scaler.inverse_transform(test_hat_vae.numpy())

print(f"VAE d=2 训练MSE={vae_train_mse:.4f}，测试MSE={vae_test_mse:.4f}")

fig = make_subplots(
    rows=1, cols=3,
    specs=[[{"type": "scatter"}, {"type": "scatter3d"}, {"type": "scatter3d"}]],
    subplot_titles=[
        "VAE d=2 latent：训练后验均值 vs 标准正态采样<br><sup>蓝点 = train μ(x)，红点 = z ~ N(0, I)</sup>",
        "VAE d=2 测试集：原始点 vs 重建点<br><sup>看规则化 latent 后，held-out reconstruction 还剩多少误差</sup>",
        "VAE d=2 prior sample decode<br><sup>红点 = z~N(0,I) 解码；观察是否比普通 AE 更贴近流形</sup>",
    ],
    horizontal_spacing=0.06,
)

fig.add_trace(go.Scatter(
    x=train_mu_np[:, 0], y=train_mu_np[:, 1],
    mode='markers',
    marker=dict(size=3, color='steelblue', opacity=0.72),
    name='训练集后验均值 μ(x)',
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=z_prior[:, 0].numpy(), y=z_prior[:, 1].numpy(),
    mode='markers',
    marker=dict(size=3, color='tomato', opacity=0.48),
    name='标准正态采样 z',
), row=1, col=1)
fig.update_xaxes(title_text='z₁', row=1, col=1)
fig.update_yaxes(title_text='z₂', row=1, col=1)

fig.add_trace(go.Scatter3d(
    x=test_np[:, 0], y=test_np[:, 1], z=test_np[:, 2],
    mode='markers',
    marker=dict(size=1.8, color='lightsteelblue', opacity=0.25),
    name='测试集原始点',
), row=1, col=2)
fig.add_trace(go.Scatter3d(
    x=test_hat_vae_np[:, 0], y=test_hat_vae_np[:, 1], z=test_hat_vae_np[:, 2],
    mode='markers',
    marker=dict(size=2.1, color='darkorange', opacity=0.78),
    name='VAE 测试集重建点',
), row=1, col=2)

fig.add_trace(go.Scatter3d(
    x=train_np[:, 0], y=train_np[:, 1], z=train_np[:, 2],
    mode='markers',
    marker=dict(size=1.5, color='lightsteelblue', opacity=0.22),
    name='训练流形参考',
), row=1, col=3)
fig.add_trace(go.Scatter3d(
    x=sampled_vae[:, 0], y=sampled_vae[:, 1], z=sampled_vae[:, 2],
    mode='markers',
    marker=dict(size=2.1, color='tomato', opacity=0.78),
    name='VAE prior 解码点',
), row=1, col=3)

fig.update_layout(
    scene2=dict(xaxis=dict(title='x'), yaxis=dict(title='h'), zaxis=dict(title='y')),
    scene3=dict(xaxis=dict(title='x'), yaxis=dict(title='h'), zaxis=dict(title='y')),
    height=560,
    showlegend=True,
    legend=dict(orientation='h', x=0.5, y=0.02, xanchor='center', yanchor='bottom'),
    title=dict(text='VAE（d=2）采样对比：规则 latent 是否更适合直接采样？', x=0.5, font=dict(size=14)),
)
fig.show()


#### 小结

- 普通 AE 只学会了“把训练样本编码到某些 latent 点，再从这些点重建回来”，并没有约束整个 latent 空间的形状。
- 因此，在 latent 包围盒里均匀随机采样时，很多 z 会落到训练从未覆盖的区域；decoder 只能外推，所以解码点常常偏离参考流形。
- VAE 在重建损失之外加入 KL 散度，把 latent 分布拉向 N(0, I)，因此随机采样 z ~ N(0,I) 比普通 AE 的包围盒采样更有依据。
- 实验上，VAE 的采样点通常会更均匀地分散在训练参考流形附近，说明规则化 latent 确实改善了采样可用性。
- 但 VAE 采样仍可能有一部分偏离流形：因为它解决的是“让 latent 更适合采样”，不是保证采样点一定完美落在真实流形上。
- 对 Swiss Roll 来说，真实内在参数域更接近 (t,h) 的矩形，而不是二维标准正态；所以 VAE 这里展示的是一种通用规则先验的收益与代价，不是对真实参数域的精确恢复。

---

### 第四步：找到分布外点的行为

**先回顾第三步的现象**

在扫描瓶颈维度时，从包围盒均匀采样的潜在点解码后，有许多红点明显偏离了 Swiss Roll 表面。原因是：第三步**跳过了编码器**，直接在潜在空间 z 采样，包围盒角落里没有训练数据对应的 z 值，解码器拿到一个从未见过的 z，照样给出输出，结果偏离了流形。

**第四步把 OOD 搬到输入空间**

生成 100 个**不在 Swiss Roll 上**的随机三维点，走完整的编码-解码流程，与第三步对比：

| 步骤 | 操作流程 | OOD 发生在 | 结果 |
|:---|:---|:---|:---|
| 第三步 | 直接在 z 空间采样 → 仅解码 | 潜在空间：z 值没有训练支撑 | 解码偏离流形 |
| 第四步 | OOD 三维点 → 编码 → 解码 | 输入空间：三维点本身没见过 | 编码器把输入压到已学到的 latent 区域，解码通常回到流形附近 |

这是这章最核心的实验之一：亲手验证「**网络没有"这个点不在我的流形上"的警报**」。


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

model_d2 = results[2]['model']
model_d2.eval()

# 先在原始 3D 坐标系中生成 OOD 点，再映射到模型使用的标准化坐标系
ood_raw_np = np.random.randn(100, 3) * train_np.std(axis=0)
ood_scaled_np = scaler.transform(ood_raw_np)
ood_data = torch.tensor(ood_scaled_np, dtype=torch.float32)

with torch.no_grad():
    ood_hat_scaled, ood_z = model_d2(ood_data)
    train_hat_d2, _ = model_d2(train_data)

ood_hat_np = scaler.inverse_transform(ood_hat_scaled.numpy())

# 每样本重建误差：统一在模型训练使用的标准化坐标系中比较
loss_fn = nn.MSELoss(reduction='none')
train_errors = loss_fn(train_hat_d2, train_data).mean(dim=1).numpy()
ood_errors   = loss_fn(ood_hat_scaled, ood_data).mean(dim=1).numpy()

print(f"训练集平均重建误差（标准化坐标）：{train_errors.mean():.4f} ± {train_errors.std():.4f}")
print(f"分布外点平均重建误差（标准化坐标）：{ood_errors.mean():.4f} ± {ood_errors.std():.4f}")

# ── 三联图：OOD 原始 | OOD 重建 | 误差分布 ─────────────────────────
fig = make_subplots(
    rows=1, cols=3,
    specs=[[{"type": "scatter3d"}, {"type": "scatter3d"}, {"type": "xy"}]],
    subplot_titles=[
        "OOD 原始点（原始 3D 坐标中的随机点）",
        "OOD 点重建后（通常被拉向训练流形附近）<br><sup>橙点未必完全贴合蓝色 Swiss Roll，但会明显收缩到已学结构附近</sup>",
        "重建误差分布：训练集 vs OOD<br><sup>OOD 误差通常更高，但模型仍会输出一个看似合理的重建</sup>",
    ],
    horizontal_spacing=0.04,
)

_manifold = dict(
    x=train_np[:, 0], y=train_np[:, 1], z=train_np[:, 2],
    mode='markers',
    marker=dict(size=1.5, color='lightsteelblue', opacity=0.2),
    name='训练流形', showlegend=False,
)

# 左图：OOD 原始点
fig.add_trace(go.Scatter3d(**_manifold), row=1, col=1)
fig.add_trace(go.Scatter3d(
    x=ood_raw_np[:, 0], y=ood_raw_np[:, 1], z=ood_raw_np[:, 2],
    mode='markers', marker=dict(size=2, color='red', opacity=0.8),
    name='OOD 原始点',
), row=1, col=1)

# 中图：OOD 重建点
fig.add_trace(go.Scatter3d(**_manifold), row=1, col=2)
fig.add_trace(go.Scatter3d(
    x=ood_hat_np[:, 0], y=ood_hat_np[:, 1], z=ood_hat_np[:, 2],
    mode='markers', marker=dict(size=2, color='orange', opacity=0.85),
    name='OOD 重建点',
), row=1, col=2)

# 右图：误差分布直方图
fig.add_trace(go.Histogram(
    x=train_errors, name='训练集', opacity=0.65,
    marker_color='steelblue', nbinsx=30,
), row=1, col=3)
fig.add_trace(go.Histogram(
    x=ood_errors, name='OOD', opacity=0.65,
    marker_color='tomato', nbinsx=30,
), row=1, col=3)

_ax3d = dict(
    xaxis=dict(title="x"), yaxis=dict(title="h"), zaxis=dict(title="y")
)
fig.update_layout(
    scene=_ax3d, scene2=_ax3d,
    barmode='overlay',
    height=520, showlegend=True,
    legend=dict(x=0.72, y=0.95),
    title=dict(
        text="第四步：输入空间中的 OOD 点会被压向已学到的流形区域",
        x=0.5, font=dict(size=14),
    ),
)
fig.update_xaxes(title_text="重建误差（MSE）", row=1, col=3)
fig.update_yaxes(title_text="频数", row=1, col=3)
fig.show()


---

**问题 4 的回答：分布外点被"拉到"了哪里？**

OOD 点经过自动编码器后，重建结果通常会**明显收缩到训练流形附近**，而不是在三维空间里继续随机散布。

流程：
1. 编码器把 OOD 点投影到潜在空间的某个坐标 z
2. 解码器把 z 解码回三维，输出只能是"从流形上长出来"的点
3. 因此输出往往会回到模型熟悉的流形区域附近，但不保证完全贴在真实流形上

**第三步与第四步的区别**

两步触发 OOD 的位置和方向完全不同：

| 步骤 | 操作流程 | OOD 发生在 | 结果 |
|:---|:---|:---|:---|
| 第三步 | 直接在 z 空间采样 → **仅解码** | 潜在空间：包围盒角落的 z 没有训练支撑 | 解码偏离流形 |
| 第四步 | OOD 三维点 → **编码** → 解码 | 输入空间：三维点本身没见过 | 编码器把输入压到已学到的 latent 区域，解码通常回到流形附近 |

第三步没有走编码器，直接给解码器一个"陌生的 z"，解码器没有能力识别这个 z 不可信，照常输出，但输出是错的。  
第四步走了完整的编码-解码流程，编码器把 OOD 点压到它学过的潜在空间范围内，所以解码结果会朝流形附近收缩。

**共同本质**：两种情况下模型都没有"不知道"这个输出选项，只能给出一个看似合理的答案。置信度是潜在坐标的函数，不是"距离训练分布有多远"的函数。


---

### 第五步：用 PCA 估算内在维度

两件事同时做：
1. **PCA 方差曲线** — 找"肘部"（解释方差突然不再增加的位置）
2. **与自动编码器 MSE 曲线对比** — 看 d 增加时误差如何下降，把 AE 结果当作一个依赖模型容量和训练设置的经验信号，而不是硬性的"维度证明"

和第一步的预测对比。


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.decomposition import PCA

# ── PCA 方差分析 ──────────────────────────────────────────────────
pca = PCA()
pca.fit(train_np)

explained_ratio  = pca.explained_variance_ratio_
cumulative_ratio = np.cumsum(explained_ratio)
n_for_95         = np.searchsorted(cumulative_ratio, 0.95) + 1
pc_labels        = [f'PC{i+1}' for i in range(len(explained_ratio))]

# ── 图1：各主成分解释方差 + 累计方差曲线 ──────────────────────────
fig1 = make_subplots(
    rows=1, cols=2,
    subplot_titles=['各主成分解释方差', '累计方差曲线（找肘部）'],
    horizontal_spacing=0.12,
)

fig1.add_trace(go.Bar(
    x=pc_labels,
    y=explained_ratio,
    text=[f'{v:.3f}' for v in explained_ratio],
    textposition='outside',
    marker_color='steelblue',
    name='各 PC 方差',
), row=1, col=1)

fig1.add_trace(go.Scatter(
    x=list(range(1, len(cumulative_ratio) + 1)),
    y=cumulative_ratio,
    mode='lines+markers',
    marker=dict(size=8),
    line=dict(color='tomato', width=2),
    name='累计方差',
), row=1, col=2)

fig1.add_hline(
    y=0.95,
    line_dash='dash',
    line_color='gray',
    annotation_text='95% 阈值',
    annotation_position='top right',
    row=1, col=2,
)

fig1.update_xaxes(title_text='主成分编号', row=1, col=1)
fig1.update_yaxes(title_text='解释方差比例', range=[0, max(explained_ratio) * 1.18], row=1, col=1)
fig1.update_xaxes(title_text='主成分数量', tickvals=list(range(1, len(explained_ratio) + 1)), row=1, col=2)
fig1.update_yaxes(title_text='累计解释方差', range=[0, 1.08], row=1, col=2)
fig1.update_layout(
    height=420,
    showlegend=False,
    title=dict(text='PCA 方差分析：Swiss Roll 的线性维度估计', x=0.5, font=dict(size=14)),
)
fig1.show()

print(f"各主成分解释方差：{explained_ratio.round(4)}")
print(f"累计解释方差：    {cumulative_ratio.round(4)}")
print(f"达到 95% 所需主成分数：{n_for_95}")

# ── 图2：自动编码器 MSE 曲线（与 PCA 对比）────────────────────────
ds       = list(results.keys())
test_mse = [results[d]['test_mse'] for d in ds]
deltas   = np.diff(test_mse)
largest_drop_idx = int(np.argmin(deltas))

fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    x=ds, y=test_mse,
    mode='lines+markers',
    marker=dict(size=8),
    line=dict(color='tomato', width=2),
    name='测试 MSE',
))
fig2.add_vline(
    x=2,
    line_dash='dash',
    line_color='steelblue',
    annotation_text='真实内在维度 d=2',
    annotation_position='top right',
)
fig2.update_layout(
    xaxis=dict(title='瓶颈维度 d', tickvals=ds),
    yaxis=dict(title='测试集重建误差（MSE）'),
    height=380,
    title=dict(text='AE 测试 MSE vs 瓶颈维度（经验曲线）', x=0.5, font=dict(size=14)),
    showlegend=True,
)
fig2.show()

print(f"\nPCA 估计内在维度（95% 方差）：{n_for_95}")
print(f"AE 误差下降最大的一步：d={ds[largest_drop_idx]}→{ds[largest_drop_idx + 1]}")
print("说明：这里的 AE 曲线只能作为经验信号；是否出现拐点也会受模型容量、训练轮数和优化设置影响。")


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── 按 t 值分段计算测试集重建误差 ────────────────────────────────
# 验证：外圈（t 大，采样稀疏）重建误差是否高于内圈（t 小，采样密集）

test_t = intrinsic_t[split:]          # 测试集的内在坐标 t
n_bins = 6
bins   = np.linspace(test_t.min(), test_t.max(), n_bins + 1)

loss_fn = nn.MSELoss(reduction='none')

# 对 d=1, 2, 3 分别计算分段 MSE
seg_results = {}
for d in [1, 2, 3]:
    model = results[d]['model']
    model.eval()
    with torch.no_grad():
        hat, _ = model(test_data)
        per_sample_mse = loss_fn(hat, test_data).mean(dim=1).numpy()

    bin_means = []
    bin_counts = []
    for i in range(n_bins):
        mask = (test_t >= bins[i]) & (test_t < bins[i + 1])
        if mask.sum() > 0:
            bin_means.append(per_sample_mse[mask].mean())
            bin_counts.append(mask.sum())
        else:
            bin_means.append(np.nan)
            bin_counts.append(0)

    seg_results[d] = {'means': bin_means, 'counts': bin_counts}

# ── 区间标签 ─────────────────────────────────────────────────────
bin_labels = [
    f"{bins[i]:.1f}–{bins[i+1]:.1f}<br>({seg_results[2]['counts'][i]} 点)"
    for i in range(n_bins)
]
bin_centers = 0.5 * (bins[:-1] + bins[1:])  # 用于颜色渐变

# ── 双图：左=分段 MSE 柱状图，右=MSE 随 t 的散点（精细视图）──────
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        "按 t 分段的平均重建误差（d=2）<br>"
        "<sup>可观察是否存在外圈更高的趋势，但不一定严格单调</sup>",
        "每个测试样本的重建误差 vs t（d=1, 2, 3）<br>"
        "<sup>看误差随 t 的变化趋势，而不是预设单调性</sup>",
    ],
    horizontal_spacing=0.12,
)

# 左图：d=2 分段均值柱状图（按 t 大小着色）
fig.add_trace(go.Bar(
    x=bin_labels,
    y=seg_results[2]['means'],
    marker=dict(
        color=bin_centers,
        colorscale='Viridis',
        showscale=True,
        colorbar=dict(title=dict(text="t 中心值"), x=0.42,
                      len=0.75, thickness=12),
    ),
    name='d=2 分段MSE',
), row=1, col=1)
fig.update_xaxes(title_text="t 区间（内圈 → 外圈）", row=1, col=1)
fig.update_yaxes(title_text="平均重建误差（MSE）", row=1, col=1)

# 右图：d=1, 2, 3 逐样本散点（半透明，看趋势）
colors = {1: 'tomato', 2: 'steelblue', 3: 'seagreen'}
for d in [1, 2, 3]:
    model = results[d]['model']
    model.eval()
    with torch.no_grad():
        hat, _ = model(test_data)
        per_sample = loss_fn(hat, test_data).mean(dim=1).numpy()
    fig.add_trace(go.Scatter(
        x=test_t, y=per_sample,
        mode='markers',
        marker=dict(size=3, color=colors[d], opacity=0.4),
        name=f'd={d}',
    ), row=1, col=2)
fig.update_xaxes(title_text="内在坐标 t（内圈小，外圈大）", row=1, col=2)
fig.update_yaxes(title_text="重建误差（MSE）", row=1, col=2)

fig.update_layout(
    height=460,
    title=dict(
        text="稀疏区域验证：外圈是否往往更难重建？",
        x=0.5, font=dict(size=13),
    ),
)
fig.show()

# ── 打印分段统计 ──────────────────────────────────────────────────
print(f"\n{'t 区间':>18} | {'点数':>6} | {'d=1 MSE':>10} | {'d=2 MSE':>10} | {'d=3 MSE':>10}")
print("─" * 62)
for i in range(n_bins):
    label = f"{bins[i]:.2f}–{bins[i+1]:.2f}"
    cnt   = seg_results[2]['counts'][i]
    vals  = [seg_results[d]['means'][i] for d in [1, 2, 3]]
    print(f"{label:>18} | {cnt:>6} | {vals[0]:>10.4f} | {vals[1]:>10.4f} | {vals[2]:>10.4f}")


---

### 第六步：把 `t` 拆成相位来观察误差

上一步按 `t` 分段时，如果误差不是简单地随 `t` 单调上升，而是出现"增大一段后又从较低水平重新开始增大"的模式，那么一个自然猜测是：**这里不只是在看外圈是否更稀疏，也混入了 Swiss Roll 的角度相位效应**。

原因是 Swiss Roll 的参数化里，`t` 同时决定了：
- 半径随 `t` 增大
- 极角也随 `t` 一起旋转

因此，按 `t` 分段并不是只在比较"离中心多远"，也在比较"卷轴转到了哪个方向"。下面把 `t` 拆成：
- `phase = t mod 2π`：看误差是否随相位重复
- `turn = floor((t - t_min) / 2π)`：粗略表示处在第几圈

如果误差主要沿 `phase` 呈现周期性，而不是只随半径增大，就说明当前残差里混有明显的相位相关结构。


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── d=2 测试误差按相位分解 ───────────────────────────────────────
model_d2 = results[2]['model']
model_d2.eval()

with torch.no_grad():
    hat_d2, _ = model_d2(test_data)
    per_sample_d2 = loss_fn(hat_d2, test_data).mean(dim=1).numpy()

phase    = np.mod(test_t, 2 * np.pi)
turn_idx = np.floor((test_t - test_t.min()) / (2 * np.pi)).astype(int)

n_phase_bins  = 8
phase_bins    = np.linspace(0, 2 * np.pi, n_phase_bins + 1)
phase_centers = 0.5 * (phase_bins[:-1] + phase_bins[1:])

phase_means  = []
phase_counts = []
for i in range(n_phase_bins):
    if i == n_phase_bins - 1:
        mask = (phase >= phase_bins[i]) & (phase <= phase_bins[i + 1])
    else:
        mask = (phase >= phase_bins[i]) & (phase <  phase_bins[i + 1])
    phase_means.append(float(per_sample_d2[mask].mean()) if mask.sum() > 0 else float('nan'))
    phase_counts.append(int(mask.sum()))

unique_turns = np.unique(turn_idx)
heatmap_z = np.full((len(unique_turns), n_phase_bins), np.nan)
for r, turn in enumerate(unique_turns):
    for c in range(n_phase_bins):
        if c == n_phase_bins - 1:
            mask = (turn_idx == turn) & (phase >= phase_bins[c]) & (phase <= phase_bins[c + 1])
        else:
            mask = (turn_idx == turn) & (phase >= phase_bins[c]) & (phase <  phase_bins[c + 1])
        if mask.sum() > 0:
            heatmap_z[r, c] = per_sample_d2[mask].mean()

# ── 双图：左=相位散点+均值折线，右=圈数×相位热图 ──────────────────
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        'd=2 误差 vs 相位<br><sup>蓝点 = 每个测试样本；红线 = 各相位区间均值</sup>',
        '不同圈数 × 相位的平均误差（d=2）<br><sup>颜色深 = 误差高；横向看相位效应是否跨圈一致</sup>',
    ],
    horizontal_spacing=0.14,
)

# 左图：逐样本散点
fig.add_trace(go.Scatter(
    x=phase, y=per_sample_d2,
    mode='markers',
    marker=dict(size=4, color='steelblue', opacity=0.35),
    name='逐样本误差',
), row=1, col=1)

# 左图：区间均值折线
fig.add_trace(go.Scatter(
    x=phase_centers, y=phase_means,
    mode='lines+markers',
    marker=dict(size=7, color='tomato'),
    line=dict(color='tomato', width=2),
    name='区间均值',
), row=1, col=1)

fig.update_xaxes(title_text='phase = t mod 2π', range=[0, 2 * np.pi], row=1, col=1)
fig.update_yaxes(title_text='重建误差（MSE）', row=1, col=1)

# 右图：热图（圈数 × 相位）
x_labels = [f'{x:.2f}' for x in phase_centers]
y_labels  = [str(t) for t in unique_turns]

fig.add_trace(go.Heatmap(
    z=heatmap_z,
    x=x_labels,
    y=y_labels,
    colorscale='Viridis',
    colorbar=dict(title=dict(text='平均 MSE'), x=1.02, len=0.85, thickness=14),
    showscale=True,
    name='MSE 热图',
    hovertemplate='相位中心=%{x}<br>圈数=%{y}<br>MSE=%{z:.4f}<extra></extra>',
), row=1, col=2)

fig.update_xaxes(title_text='相位区间中心值（t mod 2π）', row=1, col=2)
fig.update_yaxes(title_text='圈数索引', row=1, col=2)

fig.update_layout(
    height=460,
    title=dict(text='第六步：d=2 误差的相位结构分析', x=0.5, font=dict(size=14)),
    showlegend=True,
    legend=dict(orientation='v', x=0.44, y=0.98, xanchor='right', yanchor='top'),
)
fig.show()

print("\n按相位分段的平均重建误差（d=2）")
print(f"{'phase 区间':>18} | {'点数':>6} | {'平均 MSE':>10}")
print("─" * 46)
for i in range(n_phase_bins):
    label = f"{phase_bins[i]:.2f}–{phase_bins[i+1]:.2f}"
    print(f"{label:>18} | {phase_counts[i]:>6} | {phase_means[i]:>10.4f}")


---

### 检验标准三问：用自己的实验数据回答

**1. 压缩低于流形内在维度（d < 2），代价是什么？在哪里出现？**

d=1 时 MSE 明显高于 d=2：代价是重建误差升高。更具体地说，高度维度 `h` 的信息被丢弃，高度不同但卷轴位置相同的点会被重建到同一个三维位置，产生"折叠错误"。代价不是随机分布的，而是集中在需要两个坐标才能区分的区域。

**2. 分布外的点如何被"幻觉式处理"？**

OOD 点（随机三维高斯噪声）经过 AE 后，重建结果通常会**被拉回训练流形附近**，而不是继续随机散布。它们不一定全部贴在真实 Swiss Roll 表面，但会明显收缩到模型熟悉的区域。这正是幻觉的几何机制：编码器把 OOD 点投影到潜在空间，解码器只能输出"从流形上长出来"的点，没有任何机制能表达"这个点不在我的流形上"。

**3. 稀疏区域的重建误差是否更高？**

Swiss Roll 外圈（t 值大）三维点间距更大、采样密度低；但 `t` 同时也控制卷轴的相位旋转。因此，按 `t` 分段看到的误差模式，可能同时混合了"外圈更稀疏"与"不同相位更难拟合"两种效应。新增的 `t mod 2π` 分析可以帮助判断误差是否带有明显的周期性结构。

---

**问题 5 的回答：PCA 估计的内在维度 vs AE 的瓶颈维度**

| 方法 | 估计维度 | 原因 |
|:---:|:---:|:---|
| 真实生成过程 | **2**（t 和 h）| 两个独立参数 |
| 自动编码器经验拐点 | **≈ 2** | d=1→2 的误差下降最大，说明二维表示已能抓住大部分结构 |
| PCA（线性）| **可能 3** | 非线性卷轴让线性方差分解更吃亏 |

**PCA 高估的根源**：Swiss Roll 在三维空间里的卷曲嵌入是非线性的，PCA 无法把这张卷起来的曲面"展开"，因此往往需要更多主成分。当前统一模型下，AE 在 d=2 已明显优于 d=1，说明二维表示很有意义；但 d=3 还能继续把误差压低，因此这条曲线更适合作为经验上的"肘部线索"，而不是对真实内在维度的严格证明。
